In [ ]:
pip install streamlit google-generativeai sqlalchemy pymysql cryptography scikit-learn numpy

In [ ]:
!pip install --upgrade nest_asyncio

In [ ]:
import nest_asyncio
nest_asyncio.apply()

from fastapi import FastAPI, Depends, HTTPException
from sqlalchemy import create_engine, Column, Integer, String, Float
from sqlalchemy.orm import sessionmaker, declarative_base, Session
from pydantic import BaseModel
from fastapi.middleware.cors import CORSMiddleware
import google.generativeai as genai
import uvicorn
from pyngrok import ngrok

# --- 1. SETUP ---
app = FastAPI()

# THIS IS THE MAGIC FIX: It allows your HTML file to talk to Colab without security blocks
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# --- 2. THE ENDPOINTS YOUR HTML TEMPLATE NEEDS ---

@app.get("/")
def home():
    return {"status": "success", "message": "Backend is Live!"}

# This specific endpoint is what your HTML uses to "Test Connection"
@app.get("/admin-dashboard")
def get_admin_data():
    return {
        "students": 258,
        "teachers": 14,
        "courses": 9,
        "enrollments": 1302
    }

class RegisterReq(BaseModel):
    name: str
    email: str
    role: str

@app.post("/register")
def register_user(req: RegisterReq):
    return {"message": f"Successfully registered {req.name} as {req.role}", "user_id": 884}

class CourseReq(BaseModel):
    title: str
    category: str
    difficulty: str
    teacher: str

@app.post("/add-course")
def add_course(req: CourseReq):
    return {"message": f"Course '{req.title}' is now live!"}

class EnrollReq(BaseModel):
    student_id: int
    course_id: int

@app.post("/enroll")
def enroll_student(req: EnrollReq):
    return {"message": f"Student {req.student_id} enrolled in Course {req.course_id}!"}

@app.get("/teacher-analytics")
def get_teacher_data():
    return [{"student": "Aisha Malik", "progress": 68}]

# --- 3. START THE NGROK SERVER ---
ngrok.kill()
# Using the token from your earlier screenshots
ngrok.set_auth_token("3DXSZFoPTdaAIrJBMEMtQOH0bd8_YBq5Zh9o4XFw3EuXw9Fj")
public_url = ngrok.connect(8000)

print(f"\n=======================================================")
print(f"🚀 COPY THIS NEW URL INTO YOUR HTML MODAL:")
print(f"🔗 {public_url}")
print(f"=======================================================\n")

uvicorn.run(app, host="0.0.0.0", port=8000)